In [2]:

# 03test_feature_engeneering.py
import pandas as pd
import numpy as np
import os

# --- 1. 定義とパス設定 ---
PREDICT_DIR = '../data/processed_test/'
cleaned_data_path = os.path.join(PREDICT_DIR, 'test_cleaned.parquet')
processed_file = 'test_features.parquet'
processed_path = os.path.join(PREDICT_DIR, processed_file)


# --- 2. データの読み込み ---
print("2. クレンジング済みデータを読み込みます。")
try:
    df_test = pd.read_parquet(cleaned_data_path)
except FileNotFoundError:
    print(f"エラー: {cleaned_data_path} が見つかりません。test01 を先に実行してください。")
    exit()

2. クレンジング済みデータを読み込みます。


In [3]:
df_test.columns

Index(['ID', '市区町村コード', '都道府県名', '市区町村名', '地区名', '最寄駅：名称', '最寄駅：距離（分）', '間取り',
       '面積（㎡）', '建築年', '建物の構造', '用途', '今後の利用目的', '都市計画', '建ぺい率（％）', '容積率（％）',
       '取引時点', '改装', '取引の事情等'],
      dtype='object')

#　既存データから特徴量作成

In [4]:

# '建築年'の処理関数
def convert_wareki_to_seireki(wareki):
    if pd.isna(wareki) or wareki is None:
        return np.nan
    
    wareki = str(wareki).strip()
    
    # '戦前'を1945に変換
    if wareki == '戦前':
        return 1945.0
        
    # 和暦変換ロジック
    try:
        if '昭和' in wareki: return 1925 + int(wareki.replace('昭和', '').replace('年', '').replace('元', '1'))
        elif '平成' in wareki: return 1988 + int(wareki.replace('平成', '').replace('年', '').replace('元', '1'))
        elif '令和' in wareki: return 2018 + int(wareki.replace('令和', '').replace('年', '').replace('元', '1'))
        elif '大正' in wareki: return 1911 + int(wareki.replace('大正', '').replace('年', '').replace('元', '1'))
        elif '明治' in wareki: return 1867 + int(wareki.replace('明治', '').replace('年', '').replace('元', '1'))
        # 西暦の場合はそのまま返す (元のデータ型を float に統一)
        return float(wareki)
    except:
        return np.nan # 変換できないものは NaN

# '取引時点'の処理
if '取引時点' in df_test.columns:
    # 四半期情報を使わず、年の数値のみを抽出
    df_test['取引時点_年'] = df_test['取引時点'].str.extract(r'(\d{4})').astype(float)
else:
    df_test['取引時点_年'] = np.nan

# '建築年'を西暦に変換
if '建築年' in df_test.columns:
    df_test['建築年_西暦'] = df_test['建築年'].apply(convert_wareki_to_seireki)
else:
    df_test['建築年_西暦'] = np.nan

# 1. 築年数_欠損フラグの追加
df_test['築年数_欠損'] = df_test['建築年_西暦'].isna().astype(int)

# 2. 取引時点での築年数の追加 (NaNはNaNのまま保持)
df_test['取引時点での築年数'] = df_test['取引時点_年'] - df_test['建築年_西暦']

In [74]:
df_test['取引の事情等'].unique()

array([None, '調停・競売等', '関係者間取引', '瑕疵有りの可能性', 'その他事情有り',
       '他の権利・負担付き、調停・競売等'], dtype=object)

In [5]:
# 03test_feature_engeneering.py (取引事情ダミー変数作成部分)

# ------------------------------------------------------------------------------
# 3.4. 取引事情ダミー変数の作成と統合 (モデル要求の3列を生成)
# ------------------------------------------------------------------------------
print("3.4. 取引事情ダミー変数を作成・統合します。")

# 1. 欠損値の処理: 訓練時と同様に、欠損値 (None) を明示的なカテゴリとして扱う
# NaNを 'None' という文字列に置き換える。これにより、get_dummiesで一つのカテゴリとして扱われる。
ORIGINAL_COL = '取引の事情等'
if ORIGINAL_COL in df_test.columns:
    df_test[ORIGINAL_COL] = df_test[ORIGINAL_COL].fillna('None')
else:
    print(f"   ⚠️ 警告: 元の列 '{ORIGINAL_COL}' が df_test に存在しません。処理をスキップします。")
    # この後の処理で KeyError を避けるため、必要な3列を0で初期化して終了する
    df_test['取引の事情等_関係者間取引'] = 0
    df_test['取引の事情等_調停・競売等'] = 0
    df_test['取引の事情等_その他'] = 0
    # return # 実際にはここで関数を抜ける

    
# 2. 訓練データで確認された全てのカテゴリをリスト化
# ⚠️ 訓練データで確認された全てのカテゴリを正確に含めてください。
KNOWN_CATEGORIES_FROM_TRAIN = [
    'None', 
    '調停・競売等', 
    '関係者間取引', 
    '瑕疵有りの可能性', 
    'その他事情有り',
    '他の権利・負担付き、調停・競売等'
]

# 3. 訓練データと同じダミー変数のセットを生成
# reindexを利用して、テストデータに存在しないカテゴリ列を0で埋めるのが最も安全。
# 元の列をダミー変数に変換
df_dummies = pd.get_dummies(df_test[ORIGINAL_COL], prefix='取引の事情等', dtype=int)

# 訓練データで存在した可能性のある全ての列名を定義
required_dummy_cols_all = [f'取引の事情等_{cat}' for cat in KNOWN_CATEGORIES_FROM_TRAIN]

# reindexを使用して、不足している列を0で埋める
df_dummies = df_dummies.reindex(columns=required_dummy_cols_all, fill_value=0)

# 4. モデルが要求する最終的な3列に統合

# A) 取引の事情等_調停・競売等: '調停・競売等' のダミー列
df_test['取引の事情等_調停・競売等'] = df_dummies['取引の事情等_調停・競売等']

# B) 取引の事情等_関係者間取引: '関係者間取引' のダミー列
df_test['取引の事情等_関係者間取引'] = df_dummies['取引の事情等_関係者間取引']


# C) 取引の事情等_その他: 残りの複雑な事情を合計
# 'その他' に含めるカテゴリ（訓練時の定義に基づく）
other_categories = [
    '瑕疵有りの可能性', 
    'その他事情有り',
    '他の権利・負担付き、調停・競売等'
]
other_cols = [f'取引の事情等_{cat}' for cat in other_categories]

# 合計列を作成し、df_testに追加
df_test['取引の事情等_その他'] = df_dummies[other_cols].sum(axis=1)


# 5. 中間ダミー列のクリーンアップ（オプション）
# 元の df_test に結合した中間ダミー列を削除（メモリ効率のため）
# df_test = df_test.drop(columns=required_dummy_cols_all, errors='ignore')

print("   ✅ 取引事情ダミー変数3列の作成と統合が完了しました。")

3.4. 取引事情ダミー変数を作成・統合します。
   ✅ 取引事情ダミー変数3列の作成と統合が完了しました。


In [6]:
df_test

,ID,市区町村コード,都道府県名,市区町村名,地区名,最寄駅：名称,最寄駅：距離（分）,間取り,面積（㎡）,建築年,...,取引時点,改装,取引の事情等,取引時点_年,建築年_西暦,築年数_欠損,取引時点での築年数,取引の事情等_調停・競売等,取引の事情等_関係者間取引,取引の事情等_その他
0,1000000,1101,北海道,札幌市中央区,旭ケ丘,円山公園,26.0,３ＬＤＫ,75,昭和64年,...,2020年第２四半期,未改装,None,2020.0,1989.0,0,31.0,0,0,0
1,1000056,1101,北海道,札幌市中央区,大通西,西１１丁目,1.0,２ＬＤＫ,55,平成28年,...,2020年第１四半期,未改装,None,2020.0,2016.0,0,4.0,0,0,0
2,1000108,1101,北海道,札幌市中央区,大通西,西１８丁目,2.0,１Ｒ,15,昭和64年,...,2020年第２四半期,未改装,None,2020.0,1989.0,0,31.0,0,0,0
3,1000109,1101,北海道,札幌市中央区,大通西,西１８丁目,2.0,１ＬＤＫ,45,平成3年,...,2020年第２四半期,改装済,None,2020.0,1991.0,0,29.0,0,0,0
4,1000110,1101,北海道,札幌市中央区,大通西,西１８丁目,3.0,１Ｒ,20,昭和56年,...,2020年第２四半期,None,None,2020.0,1981.0,0,39.0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19461,47003828,47208,沖縄県,浦添市,牧港,None,NaN,４ＬＤＫ,80,平成31年,...,2020年第１四半期,未改装,None,2020.0,2019.0,0,1.0,0,0,0
19462,47003829,47208,沖縄県,浦添市,牧港,None,NaN,２ＬＤＫ,70,平成10年,...,2020年第１四半期,改装済,None,2020.0,1998.0,0,22.0,0,0,0
19463,47003880,47208,沖縄県,浦添市,港川,None,NaN,４ＬＤＫ,50,平成12年,...,2020年第１四半期,未改装,None,2020.0,2000.0,0,20.0,0,0,0
19464,47006648,47211,沖縄県,沖縄市,与儀,None,NaN,３ＬＤＫ,60,平成31年,...,2020年第１四半期,未改装,None,2020.0,2019.0,0,1.0,0,0,0


In [7]:
df_test=pd.get_dummies(df_test,columns=["改装"],dtype=int)
df_test

,ID,市区町村コード,都道府県名,市区町村名,地区名,最寄駅：名称,最寄駅：距離（分）,間取り,面積（㎡）,建築年,...,取引の事情等,取引時点_年,建築年_西暦,築年数_欠損,取引時点での築年数,取引の事情等_調停・競売等,取引の事情等_関係者間取引,取引の事情等_その他,改装_改装済,改装_未改装
0,1000000,1101,北海道,札幌市中央区,旭ケ丘,円山公園,26.0,３ＬＤＫ,75,昭和64年,...,None,2020.0,1989.0,0,31.0,0,0,0,0,1
1,1000056,1101,北海道,札幌市中央区,大通西,西１１丁目,1.0,２ＬＤＫ,55,平成28年,...,None,2020.0,2016.0,0,4.0,0,0,0,0,1
2,1000108,1101,北海道,札幌市中央区,大通西,西１８丁目,2.0,１Ｒ,15,昭和64年,...,None,2020.0,1989.0,0,31.0,0,0,0,0,1
3,1000109,1101,北海道,札幌市中央区,大通西,西１８丁目,2.0,１ＬＤＫ,45,平成3年,...,None,2020.0,1991.0,0,29.0,0,0,0,1,0
4,1000110,1101,北海道,札幌市中央区,大通西,西１８丁目,3.0,１Ｒ,20,昭和56年,...,None,2020.0,1981.0,0,39.0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19461,47003828,47208,沖縄県,浦添市,牧港,None,NaN,４ＬＤＫ,80,平成31年,...,None,2020.0,2019.0,0,1.0,0,0,0,0,1
19462,47003829,47208,沖縄県,浦添市,牧港,None,NaN,２ＬＤＫ,70,平成10年,...,None,2020.0,1998.0,0,22.0,0,0,0,1,0
19463,47003880,47208,沖縄県,浦添市,港川,None,NaN,４ＬＤＫ,50,平成12年,...,None,2020.0,2000.0,0,20.0,0,0,0,0,1
19464,47006648,47211,沖縄県,沖縄市,与儀,None,NaN,３ＬＤＫ,60,平成31年,...,None,2020.0,2019.0,0,1.0,0,0,0,0,1


In [8]:
# 03test_feature_engeneering.py (抜粋)

# ... (3.4. 取引事情ダミー変数作成の後に追加) ...

# ==============================================================================
# 3.5. 間取りの処理 (訓練データの頻度に基づいた統合とダミー変数化)
# ==============================================================================
print("3.5. 間取りの処理: 訓練データの頻度に基づきカテゴリを統合します。")

# 訓練データから得られた、残すべきカテゴリのリストを定義 🚨
# (閾値 1000 件に基づいて決定)
RETAINED_CATEGORIES = [
    '３ＬＤＫ', '１Ｋ', '２ＬＤＫ', '４ＬＤＫ', '１ＬＤＫ', '２ＤＫ', 
    '欠損値', '１ＤＫ', '３ＤＫ', '１Ｒ', 'オープンフロア', '２ＬＤＫ＋Ｓ', 
    '４ＤＫ', '２Ｋ'
]
GROUPED_COL = '間取り_grouped'
ORIGINAL_COL = '間取り'

if ORIGINAL_COL in df_test.columns:
    
    # 1. 欠損値を '欠損値' 文字列で埋める
    df_test[GROUPED_COL] = df_test[ORIGINAL_COL].fillna("欠損値")
    
    # 2. 訓練データで決定したリストに含まれないカテゴリを "その他" に置換
    df_test[GROUPED_COL] = df_test[GROUPED_COL].apply(
        lambda x: x if x in RETAINED_CATEGORIES else "その他"
    )

    # 3. ダミー変数化の準備: 訓練データで生成された全てのダミー列名を定義
    required_dummy_cols = [f'{GROUPED_COL}_{cat}' for cat in RETAINED_CATEGORIES]
    required_dummy_cols.append(f'{GROUPED_COL}_その他')
    
    # 4. ダミー変数化
    df_dummies = pd.get_dummies(df_test[GROUPED_COL], prefix=GROUPED_COL, dtype=int)
    
    # 5. reindexで列を揃え、df_testに結合
    df_dummies = df_dummies.reindex(columns=required_dummy_cols, fill_value=0)
    
    # 元のdf_testに結合
    df_test = pd.concat([df_test, df_dummies], axis=1)

    # 6. 不要な中間列を削除
    df_test = df_test.drop(columns=[ORIGINAL_COL, GROUPED_COL], errors='ignore')


3.5. 間取りの処理: 訓練データの頻度に基づきカテゴリを統合します。


In [9]:
import os
os.getcwd()
# The 'r' before the string makes it a "raw" string
os.chdir(r'C:\Users\sabri\Downloads\マンション価格予測\data')

#市区町村別の人口密度
df_mitudo = pd.read_csv('市区町村別人口密度.csv')
# 結合キーの列名
merge_key = '市区町村コード'

# 結合の実行
df_test = pd.merge(df_test, df_mitudo, on=merge_key, how='left')
df_test.info()
df_test

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19466 entries, 0 to 19465
Data columns (total 42 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   19466 non-null  int64  
 1   市区町村コード              19466 non-null  int64  
 2   都道府県名                19466 non-null  object 
 3   市区町村名                19466 non-null  object 
 4   地区名                  19463 non-null  object 
 5   最寄駅：名称               19453 non-null  object 
 6   最寄駅：距離（分）            18509 non-null  float64
 7   面積（㎡）                19466 non-null  int64  
 8   建築年                  18804 non-null  object 
 9   建物の構造                18201 non-null  object 
 10  用途                   13480 non-null  object 
 11  今後の利用目的              18439 non-null  object 
 12  都市計画                 19122 non-null  object 
 13  建ぺい率（％）              19045 non-null  float64
 14  容積率（％）               19045 non-null  float64
 15  取引時点                 19466 non-null 

,ID,市区町村コード,都道府県名,市区町村名,地区名,最寄駅：名称,最寄駅：距離（分）,面積（㎡）,建築年,建物の構造,...,間取り_grouped_欠損値,間取り_grouped_１ＤＫ,間取り_grouped_３ＤＫ,間取り_grouped_１Ｒ,間取り_grouped_オープンフロア,間取り_grouped_２ＬＤＫ＋Ｓ,間取り_grouped_４ＤＫ,間取り_grouped_２Ｋ,間取り_grouped_その他,市区町村人口密度
0,1000000,1101,北海道,札幌市中央区,旭ケ丘,円山公園,26.0,75,昭和64年,ＲＣ,...,0,0,0,0,0,0,0,0,0,10660.3
1,1000056,1101,北海道,札幌市中央区,大通西,西１１丁目,1.0,55,平成28年,ＲＣ,...,0,0,0,0,0,0,0,0,0,10660.3
2,1000108,1101,北海道,札幌市中央区,大通西,西１８丁目,2.0,15,昭和64年,ＳＲＣ,...,0,0,0,1,0,0,0,0,0,10660.3
3,1000109,1101,北海道,札幌市中央区,大通西,西１８丁目,2.0,45,平成3年,ＳＲＣ,...,0,0,0,0,0,0,0,0,0,10660.3
4,1000110,1101,北海道,札幌市中央区,大通西,西１８丁目,3.0,20,昭和56年,ＲＣ,...,0,0,0,1,0,0,0,0,0,10660.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19461,47003828,47208,沖縄県,浦添市,牧港,None,NaN,80,平成31年,ＲＣ,...,0,0,0,0,0,0,0,0,0,8721.4
19462,47003829,47208,沖縄県,浦添市,牧港,None,NaN,70,平成10年,ＲＣ,...,0,0,0,0,0,0,0,0,0,8721.4
19463,47003880,47208,沖縄県,浦添市,港川,None,NaN,50,平成12年,ＳＲＣ,...,0,0,0,0,0,0,0,0,0,8721.4
19464,47006648,47211,沖縄県,沖縄市,与儀,None,NaN,60,平成31年,ＲＣ,...,0,0,0,0,0,0,0,0,0,6658.6


In [10]:
# 03test_feature_engeneering.py (修正後の 3.7. 犯罪率の結合と欠損値補完)

# ==============================================================================
# 3.7. 犯罪率の結合と欠損値補完 (外部データ集約と平均値を使用)
# ==============================================================================
print("3.7. 犯罪率の外部データ結合を開始します。")

# 1. 犯罪率データフレームのロード
df_crime_rate = pd.read_excel('市区町村別犯罪率.xlsx') 
merge_key = '市区町村コード'

# 2. 🚨 修正: 外部データの重複を解消し、平均値で集約する 🚨
# 重複している市区町村コードに対して平均値を計算します。
df_crime_rate_agg = df_crime_rate.groupby(merge_key)['犯罪発生率'].mean().reset_index()

# 3. 外部データ全体の平均値を計算 (補完値として使用)
EXTERNAL_CRIME_MEAN = df_crime_rate_agg['犯罪発生率'].mean() 

# 4. 結合の実行
initial_rows = len(df_test)
df_test = pd.merge(
    df_test, 
    df_crime_rate_agg, # 🚨 集約済みのデータフレームを使用 🚨
    on=merge_key, 
    how='left'
)

# 5. 欠損値の補完
df_test['犯罪発生率'] = df_test['犯罪発生率'].fillna(EXTERNAL_CRIME_MEAN)

final_rows = len(df_test)
if initial_rows != final_rows:
    # 🚨 修正: 警告を出しつつ、問題が解消したかをチェック
    print(f"   ✅ 行数の問題を確認: 結合後の行数 {final_rows} は元の行数 {initial_rows} と一致しました。" if initial_rows == final_rows else f"   🚨 警告: 結合により行数が依然として変化しています！ (元: {initial_rows}, 後: {final_rows})")

print(f"   ✅ 犯罪発生率の結合と補完が完了しました。補完値: {EXTERNAL_CRIME_MEAN:.4f}")

3.7. 犯罪率の外部データ結合を開始します。
   ✅ 犯罪発生率の結合と補完が完了しました。補完値: 1.2979


In [11]:
import re
# 新しく追加する人口密度のテキストデータ
density_text = """
1東京都6,451.17
2大阪府4,603.02
3神奈川県3,816.89
4埼玉県1,929.89
5愛知県1,443.06
6千葉県1,217.00
7福岡県1,022.06
8沖縄県642.85
9兵庫県635.26
10京都府546.65
11香川県488.61
12茨城県460.79
13静岡県453.15
14滋賀県348.69
15奈良県348.18
16佐賀県322.73
17広島県320.44
18宮城県308.58
19長崎県302.75
20群馬県296.97
21三重県296.37
22栃木県293.74
23石川県262.42
24岡山県257.31
25富山県234.48
26熊本県228.92
27愛媛県224.70
28山口県209.32
29和歌山県186.18
30岐阜県180.12
31山梨県176.97
32福井県176.27
33大分県171.15
34新潟県166.79
35鹿児島県166.58
36徳島県165.27
37鳥取県151.43
38長野県146.62
39宮崎県133.35
40福島県126.40
41青森県120.76
42山形県108.42
43島根県95.62
44高知県92.32
45秋田県77.02
46岩手県74.92
47北海道64.29
"""

# --- Step 2: 人口密度データを整形してデータフレームを作成 ---

prefectures_density = []
densities = []

# テキストを行ごとに分割して処理
for line in density_text.strip().split('\n'):
    # 正規表現で都道府県名と数値を抽出
    match = re.search(r'\d+([^\d,.]+)\s*([\d,.]+)', line)
    if match:
        prefecture = match.group(1).strip()
        density_str = match.group(2)

        # コンマを削除して浮動小数点数に変換
        density_float = float(density_str.replace(',', ''))

        prefectures_density.append(prefecture)
        densities.append(density_float)

# 人口密度のデータフレームを作成
density_df = pd.DataFrame({
    '都道府県名': prefectures_density,
    '人口密度': densities
})


# --- Step 3: 最終的な結合 ---

# 坪単価が入ったdfに、さらに人口密度のデータを結合
df_test = pd.merge(df_test, density_df, on='都道府県名', how='left')
df_test


,ID,市区町村コード,都道府県名,市区町村名,地区名,最寄駅：名称,最寄駅：距離（分）,面積（㎡）,建築年,建物の構造,...,間取り_grouped_３ＤＫ,間取り_grouped_１Ｒ,間取り_grouped_オープンフロア,間取り_grouped_２ＬＤＫ＋Ｓ,間取り_grouped_４ＤＫ,間取り_grouped_２Ｋ,間取り_grouped_その他,市区町村人口密度,犯罪発生率,人口密度
0,1000000,1101,北海道,札幌市中央区,旭ケ丘,円山公園,26.0,75,昭和64年,ＲＣ,...,0,0,0,0,0,0,0,10660.3,2.11,64.29
1,1000056,1101,北海道,札幌市中央区,大通西,西１１丁目,1.0,55,平成28年,ＲＣ,...,0,0,0,0,0,0,0,10660.3,2.11,64.29
2,1000108,1101,北海道,札幌市中央区,大通西,西１８丁目,2.0,15,昭和64年,ＳＲＣ,...,0,1,0,0,0,0,0,10660.3,2.11,64.29
3,1000109,1101,北海道,札幌市中央区,大通西,西１８丁目,2.0,45,平成3年,ＳＲＣ,...,0,0,0,0,0,0,0,10660.3,2.11,64.29
4,1000110,1101,北海道,札幌市中央区,大通西,西１８丁目,3.0,20,昭和56年,ＲＣ,...,0,1,0,0,0,0,0,10660.3,2.11,64.29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19461,47003828,47208,沖縄県,浦添市,牧港,None,NaN,80,平成31年,ＲＣ,...,0,0,0,0,0,0,0,8721.4,0.94,642.85
19462,47003829,47208,沖縄県,浦添市,牧港,None,NaN,70,平成10年,ＲＣ,...,0,0,0,0,0,0,0,8721.4,0.94,642.85
19463,47003880,47208,沖縄県,浦添市,港川,None,NaN,50,平成12年,ＳＲＣ,...,0,0,0,0,0,0,0,8721.4,0.94,642.85
19464,47006648,47211,沖縄県,沖縄市,与儀,None,NaN,60,平成31年,ＲＣ,...,0,0,0,0,0,0,0,6658.6,1.00,642.85


In [12]:
df_city=pd.get_dummies(df_test, columns=['都市計画'], dtype=int)
df_test['都市計画_高価格帯'] = (
    df_city['都市計画_第１種低層住居専用地域'] +
    df_city['都市計画_第２種低層住居専用地域'] +
    df_city['都市計画_工業専用地域'] 
)

df_test['都市計画_中価格帯'] = (
    df_city['都市計画_商業地域'] + 
    df_city['都市計画_準工業地域'] +  
    df_city['都市計画_工業地域'] +  
    df_city['都市計画_第１種中高層住居専用地域'] +
    df_city['都市計画_第２種中高層住居専用地域'] +
    df_city['都市計画_第１種住居地域'] +
    df_city['都市計画_第２種住居地域'] +
    df_city['都市計画_準住居地域'] +
    df_city['都市計画_近隣商業地域'] 
)

df_test['都市計画_低価格帯'] = (
    df_city['都市計画_市街化調整区域'] +
    df_city['都市計画_市街化区域及び市街化調整区域外の都市計画区域'] 
)

In [13]:
import pandas as pd
import numpy as np
import warnings
# import joblib # 訓練データからマップをロードする場合に必要

warnings.filterwarnings('ignore')

# ------------------------------------------------------------------------------
# 3.8. 交互作用項と非線形変換の作成
# ------------------------------------------------------------------------------
print("\n" + "="*50)
print("3.8. 交互作用項と非線形変換を作成中...")
print("="*50)

# ⚠️ 処理対象を df_test に設定 ⚠️
df = df_test 

# 計算に必要な外部データ列が存在するか確認し、NaNで初期化 (安全対策)
REQUIRED_COLS = ['取引時点での築年数', '面積（㎡）', '最寄駅：距離（分）', 
                 '建ぺい率（％）', '容積率（％）', '人口密度', '市区町村人口密度']
for col in REQUIRED_COLS:
    if col not in df.columns:
        df[col] = np.nan
        # print(f"   ⚠️ 警告: '{col}' が見つからないため、NaNで初期化しました。")


# 2. 交互作用項の作成
print("\n【非線形変換と交互作用項】")

# 築年数関連（非線形変換）
df['築年数_2乗'] = df['取引時点での築年数'] ** 2
df['築年数_3乗'] = df['取引時点での築年数'] ** 3
df['築年数_log'] = np.log1p(df['取引時点での築年数'])

# 面積関連（非線形変換）
df['面積_log'] = np.log1p(df['面積（㎡）'])
df['面積_平方根'] = np.sqrt(df['面積（㎡）'])

# 2項の交互作用
df['築年数×面積'] = df['取引時点での築年数'] * df['面積（㎡）']
df['築年数×駅距離'] = df['取引時点での築年数'] * df['最寄駅：距離（分）']
df['築年数×建ぺい率'] = df['取引時点での築年数'] * df['建ぺい率（％）']
df['築年数×容積率'] = df['取引時点での築年数'] * df['容積率（％）']
df['築年数×人口密度'] = df['取引時点での築年数'] * df['人口密度']
df['面積×駅距離'] = df['面積（㎡）'] * df['最寄駅：距離（分）']
df['面積×建ぺい率'] = df['面積（㎡）'] * df['建ぺい率（％）']
df['面積×容積率'] = df['面積（㎡）'] * df['容積率（％）']
df['面積×人口密度'] = df['面積（㎡）'] * df['人口密度']

# 建ぺい率・容積率の組み合わせ
df['建築可能性'] = df['建ぺい率（％）'] * df['容積率（％）'] / 100
# ゼロ割を避けるため、分母に0.1を加える
df['容積率_建ぺい率比'] = df['容積率（％）'] / (df['建ぺい率（％）'].replace(0, 0.1) + 0.1) 
df['建ぺい率_2乗'] = df['建ぺい率（％）'] ** 2
df['容積率_2乗'] = df['容積率（％）'] ** 2

# 駅距離関連
# ゼロ割を避けるため、分母に1を加える
df['駅距離_逆数'] = 1 / (df['最寄駅：距離（分）'] + 1) 
df['駅距離_log'] = np.log1p(df['最寄駅：距離（分）'])
df['駅距離_2乗'] = df['最寄駅：距離（分）'] ** 2
df['駅距離×建ぺい率'] = df['最寄駅：距離（分）'] * df['建ぺい率（％）']
df['駅距離×容積率'] = df['最寄駅：距離（分）'] * df['容積率（％）']

# 人口密度関連
df['人口密度_log'] = np.log1p(df['人口密度'])
df['市区町村人口密度_log'] = np.log1p(df['市区町村人口密度'])
df['人口密度×建ぺい率'] = df['人口密度'] * df['建ぺい率（％）']

# 3項の交互作用（試験的）
df['築年数×面積×駅距離'] = df['取引時点での築年数'] * df['面積（㎡）'] * df['最寄駅：距離（分）']
df['面積×建ぺい率×容積率'] = df['面積（㎡）'] * df['建ぺい率（％）'] * df['容積率（％）']
print("✓ 交互作用項の作成完了")

# ========================================
# 4. 頻度エンコーディング
# ========================================

print("\n" + "="*50)
print("頻度エンコーディング中...")
print("="*50)

# 建ぺい率・容積率の頻度
df['建ぺい率_頻度'] = df['建ぺい率（％）'].map(df['建ぺい率（％）'].value_counts())
df['容積率_頻度'] = df['容積率（％）'].map(df['容積率（％）'].value_counts())
print("✓ 建ぺい率_頻度, 容積率_頻度")

# 市区町村の頻度
df['市区町村_頻度'] = df['市区町村名'].map(df['市区町村名'].value_counts())
df['市区町村_頻度_log'] = np.log1p(df['市区町村_頻度'])
print("✓ 市区町村_頻度, 市区町村_頻度_log")

# 駅名の頻度
df['駅名_頻度'] = df['最寄駅：名称'].map(df['最寄駅：名称'].value_counts())
df['駅名_頻度_log'] = np.log1p(df['駅名_頻度'])
print("✓ 駅名_頻度, 駅名_頻度_log")

# 地区名の頻度
df['地区名_頻度'] = df['地区名'].map(df['地区名'].value_counts())
df['地区名_頻度_log'] = np.log1p(df['地区名_頻度'])
print("✓ 地区名_頻度, 地区名_頻度_log")


df_test = df # 変更を df_test に反映


3.8. 交互作用項と非線形変換を作成中...

【非線形変換と交互作用項】
✓ 交互作用項の作成完了

頻度エンコーディング中...
✓ 建ぺい率_頻度, 容積率_頻度
✓ 市区町村_頻度, 市区町村_頻度_log
✓ 駅名_頻度, 駅名_頻度_log
✓ 地区名_頻度, 地区名_頻度_log


In [14]:
import pandas as pd
import os

print("\n" + "="*50)
print("4. 処理後のデータを保存")
print("="*50)

# --- 1. データフレームのサマリー表示 ---
print("--- 特徴量作成後のデータ (df_test) ---")
print(df_test.head())

# --- 2. 処理後のデータを保存 ---
# 予測用データは 'processed_test' ディレクトリに保存します。
PROCESSED_TEST_DIR = '../data/processed_test/'
PROCESSED_TEST_FILE = 'test_features.parquet' # モデル予測用の最終データ
PROCESSED_TEST_PATH = os.path.join(PROCESSED_TEST_DIR, PROCESSED_TEST_FILE)

# ディレクトリが存在しない場合は作成
os.makedirs(PROCESSED_TEST_DIR, exist_ok=True)

# Parquet形式で保存 (index=Falseでインデックスを保存しない)
df_test.to_parquet(PROCESSED_TEST_PATH, index=False)

print(f"\n✅ 特徴量作成後の予測用データを {PROCESSED_TEST_PATH} に保存しました。")
print(f"行数: {len(df_test)}, カラム数: {df_test.shape[1]}")


4. 処理後のデータを保存
--- 特徴量作成後のデータ (df_test) ---
        ID  市区町村コード 都道府県名   市区町村名  地区名 最寄駅：名称  最寄駅：距離（分）  面積（㎡）    建築年 建物の構造  \
0  1000000     1101   北海道  札幌市中央区  旭ケ丘   円山公園       26.0     75  昭和64年    ＲＣ   
1  1000056     1101   北海道  札幌市中央区  大通西  西１１丁目        1.0     55  平成28年    ＲＣ   
2  1000108     1101   北海道  札幌市中央区  大通西  西１８丁目        2.0     15  昭和64年   ＳＲＣ   
3  1000109     1101   北海道  札幌市中央区  大通西  西１８丁目        2.0     45   平成3年   ＳＲＣ   
4  1000110     1101   北海道  札幌市中央区  大通西  西１８丁目        3.0     20  昭和56年    ＲＣ   

   ... 築年数×面積×駅距離 面積×建ぺい率×容積率 建ぺい率_頻度  容積率_頻度  市区町村_頻度 市区町村_頻度_log 駅名_頻度  \
0  ...    60450.0    180000.0   122.0    15.0      169    5.135798  29.0   
1  ...      220.0   2640000.0  8247.0  1333.0      169    5.135798  15.0   
2  ...      930.0    480000.0  8247.0  3318.0      169    5.135798  30.0   
3  ...     2610.0   1440000.0  8247.0  3318.0      169    5.135798  30.0   
4  ...     2340.0    640000.0  8247.0  3318.0      169    5.135798  30.0   

   駅名_頻度_log  地区名_